In [0]:
"""Creating a function to add ingested and updated timestamp and uploading the data into the silver table. Creates the delta table if it doesnot exist. Otherwise merge the input Dataframe into the target table """

from pyspark.sql import functions as F
from delta.tables import DeltaTable
def write_to_silver(input_df,target_table,merge_condition,columns_to_update):
    #Adding the created and updated timestamp
    final_df = (input_df.withColumn("created_timestamp",F.current_timestamp()).withColumn("updated_timestamp",F.current_timestamp()))
    
    #Create the silver table if not exist
    if not spark.catalog.tableExists(target_table):
        final_df.write.format('delta').mode('overwrite').saveAsTable(target_table)
    else:
        #Assigning target table to the delta table variable
        delta_table = DeltaTable.forName(spark,target_table)
        #getting the columns to update 
        update_map = {column:f"s.{column}" for column in columns_to_update}
        #Updating only the updated timestamp
        update_map["updated_timestamp"] = "s.updated_timestamp"
        """merge the target with the source on the merge condition and update the specific columns with the latest data if the batch id is greater than the target batch id. If it was not matched then update all the data"""
        (
            delta_table.alias("t").merge(final_df.alias("s"),merge_condition)
            .whenMatchedUpdate(condition="s.batch_id>=t.batch_id",set=update_map)
            .whenNotMatchedInsertAll()
            .execute()
        )

